# E-commerce delivery network and SLA risk

## Business context

Olist is a Brazilian company that connects small businesses from across the country to the large marketplaces through a single contract (Olist, 2018). Merchants sell their products through the Olist Store, which is present in several Brazilian marketplaces, and ship the orders directly to customers through Olist's logistics partners. Once a purchase is completed, the seller is notified to prepare and dispatch the order. A single order can combine items from different sellers, each responsible for its own part of the shipment.

At purchase time, the customer receives an estimated delivery date. This date is the promise that this project treats as the SLA: it is set before any logistics step takes place, and it is the benchmark against which the customer judges the delivery. Olist's satisfaction survey is sent by email when the product arrives or when the estimated date is due, so the purchase is rated precisely around the promised date (Olist, 2018).

Keeping that promise depends on a chain that Olist does not run alone. The seller has to dispatch on time, and the logistics partner has to carry the order from the seller's state to the customer's state, often between distant regions of the country. The same promise can be kept with room to spare on one route and frequently broken on another, which is why delay is treated here as a network phenomenon, tied to the relationship between origins and destinations. If the risk of delay can be estimated at purchase time, it can inform decisions taken before the problem occurs, such as prioritizing the follow-up of the riskiest orders or revising the promise on the routes that break it most often.

The analysis uses the public dataset released by Olist, with about 100 thousand orders placed between 2016 and 2018 and anonymized commercial data (Olist, 2018).

## Source presentation and initial pre-analysis

This notebook presents the Olist source (Olist, 2018) and checks whether it supports the research question. Reading and calculation logic lives in `src/ecom_delivery_sla`; the notebook remains a reproducible narrative interface.

Central question: *Is it possible to identify delivery-delay and SLA-breach patterns from characteristics known at purchase time, seller-to-destination relationships, and the network's observed temporal structure?*

The order is the unit of analysis. Continuous delay and binary breach targets are built by the project package.

## Measurable scope and data limits

At this stage, delivery-promise fulfillment is measured directly at order level: `delivery_delay_days` compares actual and estimated delivery dates, while `sla_breach` identifies positive delay. It is also possible to describe promised lead time, actual lead time, seasonality, destination, category, seller composition, and approximate seller-to-customer distance.

These variables do not turn the dataset into complete logistics tracking. The source does not record delivery by seller or item, hub locations, the actual route, transport mode, operational delay reason, or intermediate events. Therefore, the project measures historical risk of breaking the promised delivery date; seller and geolocation indicators are proxies, not direct measurements of an individual SLA.

## Source and observable inventory

Before discussing models, it is necessary to verify which tables are available, their scale, the fields in each table, and the types pandas observes when reading the CSVs. This is a structural inventory and does not replace the analytical-table contracts.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

LANG = "en"
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ecom_delivery_sla import TABLE_FILES, schema_summary
from ecom_delivery_sla import build_supervised_orders, inventory_summary, read_olist_tables, target_summary

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'olist' / 'csv'
tables = read_olist_tables(DATA_DIR, table_names=TABLE_FILES)
display(inventory_summary(tables, language=LANG))
display(schema_summary(tables))

,tabela,linhas,colunas
0,orders,99441,8
1,customers,99441,5
2,items,112650,7
3,products,32951,9
4,payments,103886,5
5,reviews,99224,7
6,sellers,3095,4
7,geolocation,1000163,5
8,category translation,71,2


,dataset,rows,field_order,field,data_type,non_null_pct
0,orders,99441,1,order_id,object,1.000000
1,orders,99441,2,customer_id,object,1.000000
2,orders,99441,3,order_status,object,1.000000
3,orders,99441,4,order_purchase_timestamp,object,1.000000
4,orders,99441,5,order_approved_at,object,0.998391
5,orders,99441,6,order_delivered_carrier_date,object,0.982070
6,orders,99441,7,order_delivered_customer_date,object,0.970183
7,orders,99441,8,order_estimated_delivery_date,object,1.000000
8,customers,99441,1,customer_id,object,1.000000
9,customers,99441,2,customer_unique_id,object,1.000000


### Insight from the inventory

The source provides order, item, customer, seller, product, payment, review, geolocation, and translation tables. The structural table makes each field, its pandas-observed type, and its completeness visible. These are CSV-reading types (timestamps may still appear as `object` before parsing) and they do not replace the semantic data definition. This composition is sufficient to investigate delivery at order level and enrich the analysis with origin-to-destination relationships, but it is not enough to claim that every available variable was known at purchase time.

## Supervised population and initial target definition

Delivered orders with observable actual and estimated delivery dates are now retained, and delay in days and the binary breach indicator are then calculated: `sla_breach = delay > 0`. The goal is to make explicit how many observations can actually participate in supervised analysis.

In [2]:
supervised = build_supervised_orders(tables['orders'])
display(target_summary(supervised, language=LANG))

,metric,value
0,Supervised orders,96470.000000
1,SLA breaches,6534.000000
2,Breach rate,0.067731
3,Median delay (days),-11.948102
4,P95 delay (days),3.816993
5,P99 delay (days),18.939553
6,Delay > 30 days,360.000000
7,Early delivery < -30 days,2352.000000


### Insight from the target

The supervised population contains 96,470 orders, and 6,534 breach under the calendar-day rule, an overall rate of approximately 6.77%. The target is therefore imbalanced, and this definition must remain consistent across experiments. The rate describes the observed history; it is not automatically a future probability of delay.

## Current limits

This pre-analysis does not authorize causal conclusions or final modeling. The analytical table, temporal cutoff, policy for multi-seller orders, and construction of historical network metrics are still to be defined.

Until the contracts are complete, `production_eligible = false`.

Source: OLIST. *Brazilian E-Commerce Public Dataset by Olist*. Kaggle.

## References

Olist (2018). *Brazilian E-Commerce Public Dataset by Olist*. Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce